# 10.1 ? Cataluña No-Pooling Detail

This notebook is retained as a focused view of the **Cataluña no-pooling decision**. The production final model set now already uses the non-pooled choice: **Cataluña = SARIMA**. Pooled regional models remain visible as sensitivity diagnostics only.

| Series | Final model | Pooled? | 2025 validation MAPE |
|---|---|---|---|
| Nacional | SARIMA | no | 29.0% |
| Madrid | Logistic curve | no | 73.6% |
| **Cataluña** | **SARIMA** | **no** | **47.2%** |
| Andalucía | Logistic curve | no | 48.4% |
| Valencia | Gompertz curve | no | 34.2% |


## 0. Setup — load the productionized Phase 2 outputs

In [ ]:
import warnings; warnings.filterwarnings('ignore')
import numpy as np, pandas as pd
import matplotlib.pyplot as plt
import subprocess
import sys
from pathlib import Path

NOTEBOOK_DIR = Path().resolve()
REPO_ROOT    = NOTEBOOK_DIR.parent
OUT          = REPO_ROOT / 'data' / 'outputs'
FEAT         = REPO_ROOT / 'data' / 'features'

TARGETS = ['Nacional', 'Madrid', 'Cataluña', 'Andalucía', 'Valencia']
COLORS  = {'Nacional': '#FF6B35', 'Madrid': '#004E89', 'Cataluña': '#1A936F',
           'Andalucía': '#C84B31', 'Valencia': '#8E44AD'}
FINAL_MODEL_POLICY = {
    'Nacional': 'SARIMA',
    'Madrid': 'Logistic',
    'Cataluña': 'SARIMA',
    'Andalucía': 'Logistic',
    'Valencia': 'Gompertz',
}


def reload_production_outputs():
    final_df = pd.read_csv(OUT / 'metricas_final_selected.csv')
    accept_df = pd.read_csv(OUT / 'phase2_model_acceptance.csv')
    pool_exp_df = pd.read_csv(OUT / 'phase2_pooling_experiment_metrics.csv')
    sarima_grid_df = pd.read_csv(OUT / 'sarima_grid_search_results.csv')
    sarima_accept_df = pd.read_csv(OUT / 'sarima_order_acceptance.csv')
    preds_df = pd.read_csv(OUT / 'predicciones_test_2025.csv')
    forecast_df = pd.read_csv(OUT / 'forecast_24m_sarima_rf_xgb.csv')
    return final_df, accept_df, pool_exp_df, sarima_grid_df, sarima_accept_df, preds_df, forecast_df

final    = pd.read_csv(OUT / 'metricas_final_selected.csv')
accept   = pd.read_csv(OUT / 'phase2_model_acceptance.csv')
pool_exp = pd.read_csv(OUT / 'phase2_pooling_experiment_metrics.csv')
sarima_grid = pd.read_csv(OUT / 'sarima_grid_search_results.csv')
sarima_accept = pd.read_csv(OUT / 'sarima_order_acceptance.csv')
preds    = pd.read_csv(OUT / 'predicciones_test_2025.csv')
forecast = pd.read_csv(OUT / 'forecast_24m_sarima_rf_xgb.csv')
history  = pd.read_csv(FEAT / 'features_modelo_completo.csv')[['Fecha', 'Target', 'Consumo_Tm']]

SELECTED = dict(zip(final['Target'], final['Model']))
if SELECTED != FINAL_MODEL_POLICY:
    print('Detected stale notebook-generated model outputs. Rebuilding production modeling outputs...')
    subprocess.run([sys.executable, str(REPO_ROOT / 'scripts' / '05_modeling_with_cnmc.py')], check=True, cwd=REPO_ROOT)
    final, accept, pool_exp, sarima_grid, sarima_accept, preds, forecast = reload_production_outputs()
    SELECTED = dict(zip(final['Target'], final['Model']))
    if SELECTED != FINAL_MODEL_POLICY:
        raise ValueError(f'Final selected model policy mismatch after rebuild: {SELECTED}')

plt.rcParams['figure.dpi'] = 110
print('Selected model per series (production no-pooling policy):')
for t in TARGETS:
    print('  {:10s} -> {}'.format(t, SELECTED[t]))


## 1. Final selected models & how each was chosen

Same decision trail as notebook 10, except **Cataluña is forced to SARIMA** (no
pooling anywhere). The table below reflects the non-pooled SARIMA metrics for
Cataluña on the 2025 holdout.

The pooled-candidate table is still shown for reference — it illustrates why pooling
was adopted in the original selection and why removing it leaves SARIMA as the best
available alternative (Pooled Ridge explodes; other non-pooled options score worse on
both the walk-forward gate and the holdout).


In [ ]:
# 1a. Final selected models + 2025 holdout metrics
tbl = final.copy()
tbl['Pooled'] = np.where(tbl['Model'].str.startswith('Pooled'), 'yes', 'no')
tbl = tbl[['Target', 'Model', 'Pooled', 'MAE', 'RMSE', 'MAPE', 'R2']]
print('FINAL SELECTED MODELS — no-pooling variant (2025 holdout):')
print(tbl.to_string(index=False))
avg = final['MAPE'].mean()
print()
cat_mape = float(final[final['Target'] == 'Cataluña']['MAPE'])
print('Average selected 2025 MAPE: {:.1f}%  (vs 46.4% in the pooled version)'.format(avg))

# 1b. Selection decision per series (for reference)
print()
print('ORIGINAL SELECTION DECISION (phase2_model_acceptance.csv):')
acc = accept[['Target', 'Phase1_Model', 'Phase1_MAPE', 'Phase2_Proposed_Model',
              'Phase2_Proposed_MAPE', 'Selected_Model', 'Decision']]
print(acc.to_string(index=False))
print()
print('NOTE: In this variant, Cataluña uses SARIMA (non-pooled, 2025 MAPE {:.1f}%)'.format(cat_mape))
print('      instead of Pooled Random Forest (2025 MAPE 46.8%).')

# 1c. Pooled candidates for reference
print()
print('POOLED CANDIDATES, 2025 holdout MAPE (%) — shown for reference:')
pe = pool_exp.pivot_table(index='Target', columns='Model', values='MAPE')
cols = [c for c in ['Pooled Ridge', 'Pooled Random Forest', 'Pooled XGBoost'] if c in pe.columns]
print(pe[cols].round(1).to_string())
print()
print('Pooled Ridge MAPE is in the tens-of-thousands of % (unbounded blow-up) -> ruled out.')


## 1.1 SARIMA Grid-Search Robustness Check

SARIMA parameter tuning is now handled in the production script before final model selection. The grid is constrained and evaluated only inside the 2023-2024 training period using the same recursive walk-forward logic. The training-CV winner is then checked against the original default SARIMA order on the 2025 validation / acceptance period; the production order is only changed when the grid-selected order does not regress versus the default SARIMA.

The tables below are loaded directly from `sarima_grid_search_results.csv` and `sarima_order_acceptance.csv`.


In [ ]:
sarima_selected = (
    sarima_grid[sarima_grid['Selected']]
    [['Target', 'p', 'd', 'q', 'P', 'D', 'Q', 'm', 'WalkForward_MAPE', 'Successful_Folds']]
    .sort_values('Target')
    .reset_index(drop=True)
)
print('TRAINING-ONLY SARIMA GRID WINNERS')
display(sarima_selected)

sarima_accept_cols = [
    'Target', 'Default_Order', 'Default_Seasonal_Order',
    'Grid_Selected_Order', 'Grid_Selected_Seasonal_Order',
    'Default_2025_MAPE', 'Grid_Selected_2025_MAPE',
    'Production_Order', 'Production_Seasonal_Order', 'Decision'
]
print('SARIMA ORDER ACCEPTANCE FOR PRODUCTION')
display(sarima_accept[sarima_accept_cols].sort_values('Target').reset_index(drop=True))


## 2. Results — 2025 holdout fit & the 24-month forecast

Each series' selected model (Cataluña = SARIMA) against the realized 2025 values,
then the 24-month forward forecast. The dotted grey line marks the train/forecast
boundary at 2025-12.


In [ ]:
# 2025 holdout: selected model vs actuals
fig, axes = plt.subplots(3, 2, figsize=(15, 12)); axes = axes.ravel()
for i, t in enumerate(TARGETS):
    ax = axes[i]; m = SELECTED[t]
    d = preds[(preds['Target'] == t) & (preds['Model'] == m)].sort_values('Fecha').copy()
    d['date'] = pd.to_datetime(d['Fecha'])
    ax.plot(d['date'], d['Actual'], 'o-', color='black', lw=2, ms=4, label='Actual')
    ax.plot(d['date'], d['Pred'], 's--', color=COLORS[t], lw=2, ms=4, label=m)
    mape = float(final[final['Target'] == t]['MAPE'].iloc[0])
    ax.set_title('{} - {}  (2025 MAPE {:.1f}%)'.format(t, m, mape), fontsize=11, weight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylabel('Biodiesel (Tm)')
axes[-1].axis('off')
fig.suptitle('2025 holdout: selected model vs actuals (Cataluña = SARIMA, no pooling)', fontsize=13, weight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# 24-month forecast (2026-2027): history + selected-model forecast
fig, axes = plt.subplots(3, 2, figsize=(15, 12)); axes = axes.ravel()
cut = pd.to_datetime('2025-12')
for i, t in enumerate(TARGETS):
    ax = axes[i]; m = SELECTED[t]
    h = history[history['Target'] == t].sort_values('Fecha').copy()
    fc = forecast[(forecast['Target'] == t) & (forecast['Model'] == m)].sort_values('Fecha').copy()
    h['date'] = pd.to_datetime(h['Fecha']); fc['date'] = pd.to_datetime(fc['Fecha'])
    ax.plot(h['date'], h['Consumo_Tm'], '-', color='black', lw=1.8, label='Historical (2023-25)')
    ax.plot(fc['date'], fc['Forecast'], '--', color=COLORS[t], lw=2.2, label='Forecast ({})'.format(m))
    ax.axvline(cut, color='grey', ls=':', lw=1)
    ax.set_title('{} - {}'.format(t, m), fontsize=11, weight='bold')
    ax.tick_params(axis='x', rotation=45, labelsize=8); ax.legend(fontsize=8); ax.grid(alpha=0.3)
    ax.set_ylabel('Biodiesel (Tm)')
axes[-1].axis('off')
fig.suptitle('24-month biodiesel demand forecast (2026-2027) — no-pooling variant', fontsize=13, weight='bold')
fig.tight_layout(); plt.show()

In [ ]:
# Annual forecast totals (sum of each series' SELECTED model over each year)
f = forecast.merge(pd.DataFrame(SELECTED.items(), columns=['Target', 'Model']), on=['Target', 'Model'])
f['Year'] = f['Fecha'].str[:4]
annual = f.pivot_table(index='Target', columns='Year', values='Forecast', aggfunc='sum').round(0)
annual = annual.reindex(TARGETS)
print('Forecast biodiesel demand (Tm) - annual totals of the per-series selected model:')
print(annual.to_string())

## Summary & caveats ? Cataluña no-pooling detail

- **What this is:** a focused view of why Cataluña uses **SARIMA** in the final non-pooled production set.
- **Why SARIMA for Cataluña:** it is the best non-pooled model on the 2025 validation period (47.2%), essentially tied with the pooled Random Forest sensitivity case (46.8%).
- **Honest limits:** SARIMA extrapolates a growth trend and may overestimate if Cataluña demand saturates. The pooled Random Forest gave a more conservative plateau, but it is not used in the final no-pooling deliverable.
- **Provenance:** all numbers are read from `data/outputs/`; this notebook re-fits nothing.
